# 批归一化 (Batch Normalization) 作用探究

本笔记本用于探究 Batch Normalization 在深度神经网络中的作用，特别是观察其如何维持网络各隐藏层特征方差的稳定。

- **Cell 1**: 环境准备与数据集加载
- **Cell 2**: 未添加 Normalization 的多层网络训练，记录每 Epoch 后各层特征方差并可视化
- **Cell 3**: 新增 Cell - 添加 Normalization (BatchNorm1d) 的网络训练，保持网络结构不变，同样记录每 Epoch 后各层特征方差并对比可视化

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# 固定随机种子，保证实验复现性
torch.manual_seed(42)

# 设置字体以支持中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 定义图像预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# 加载 MNIST 数据集
train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset=train_set, batch_size=128, shuffle=True)
test_loader  = DataLoader(dataset=test_set, batch_size=256, shuffle=False)

print(f'训练集大小：{len(train_set)} 张')
print(f'测试集大小：{len(test_set)} 张')

In [ ]:
# === Cell 2: 无 Normalization 的网络训练与特征方差统计 ===

class DeepMLP_NoBN(nn.Module):
    def __init__(self):
        super(DeepMLP_NoBN, self).__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 32)
        self.fc5 = nn.Linear(32, 10)

    def forward_features(self, x):
        x = x.view(-1, 784)
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        h3 = F.relu(self.fc3(h2))
        h4 = F.relu(self.fc4(h3))
        out = self.fc5(h4)
        return out, [h1, h2, h3, h4]

def get_layer_variances(model, data_loader):
    """计算各隐藏层特征在整个测试集上的方差"""
    model.eval()
    layer_feats = [[] for _ in range(4)]
    with torch.no_grad():
        for xb, _ in data_loader:
            _, feats = model.forward_features(xb)
            for i, f in enumerate(feats):
                layer_feats[i].append(f)
    # 计算各个隐藏层的全特征方差
    variances = [torch.cat(feats_list, dim=0).var().item() for feats_list in layer_feats]
    return variances

# 初始化无 BN 的模型及优化器
torch.manual_seed(42)
model_nobn = DeepMLP_NoBN()
optimizer_nobn = optim.SGD(model_nobn.parameters(), lr=0.1)
loss_fn = nn.CrossEntropyLoss()

# 记录无 BN 网络每 epoch 后各层的特征方差
variances_nobn_history = []
epochs = 5

print("=== 开始训练未添加 Normalization 的网络 ===")
for epoch in range(epochs):
    model_nobn.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        optimizer_nobn.zero_grad()
        logits, _ = model_nobn.forward_features(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer_nobn.step()
        running_loss += loss.item() * yb.size(0)

    # 统计本 Epoch 结束后每层特征的方差
    vars_epoch = get_layer_variances(model_nobn, test_loader)
    variances_nobn_history.append(vars_epoch)
    
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_set):.4f} | "
          f"特征方差: Layer1={vars_epoch[0]:.4f}, Layer2={vars_epoch[1]:.4f}, Layer3={vars_epoch[2]:.4f}, Layer4={vars_epoch[3]:.4f}")

# 绘制未添加 Normalization 时各层特征方差曲线
plt.figure(figsize=(8, 5))
for layer_idx in range(4):
    vars_layer = [epoch_vars[layer_idx] for epoch_vars in variances_nobn_history]
    plt.plot(range(1, epochs + 1), vars_layer, marker='o', linewidth=2, label=f'Layer {layer_idx+1}')

plt.xlabel('Epoch（轮数）')
plt.ylabel('Feature Variance（特征方差）')
plt.title('未添加 Normalization：每 Epoch 后各层特征方差变化')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

In [ ]:
# === Cell 3: 新增 Cell - 添加 Normalization (BatchNorm1d) 的网络训练与特征方差统计 ===

class DeepMLP_WithBN(nn.Module):
    def __init__(self):
        super(DeepMLP_WithBN, self).__init__()
        self.fc1 = nn.Linear(784, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.fc4 = nn.Linear(64, 32)
        self.bn4 = nn.BatchNorm1d(32)
        self.fc5 = nn.Linear(32, 10)

    def forward_features(self, x):
        x = x.view(-1, 784)
        h1 = F.relu(self.bn1(self.fc1(x)))
        h2 = F.relu(self.bn2(self.fc2(h1)))
        h3 = F.relu(self.bn3(self.fc3(h2)))
        h4 = F.relu(self.bn4(self.fc4(h3)))
        out = self.fc5(h4)
        return out, [h1, h2, h3, h4]

# 初始化添加了 BN 的模型及优化器（其他超参数和随机种子与无 BN 网络完全一致）
torch.manual_seed(42)
model_bn = DeepMLP_WithBN()
optimizer_bn = optim.SGD(model_bn.parameters(), lr=0.1)

# 记录含 BN 网络每 epoch 后各层的特征方差
variances_bn_history = []

print("=== 开始训练添加了 Batch Normalization 的网络 ===")
for epoch in range(epochs):
    model_bn.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        optimizer_bn.zero_grad()
        logits, _ = model_bn.forward_features(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer_bn.step()
        running_loss += loss.item() * yb.size(0)

    # 统计本 Epoch 结束后每层特征的方差
    vars_epoch = get_layer_variances(model_bn, test_loader)
    variances_bn_history.append(vars_epoch)
    
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_set):.4f} | "
          f"特征方差: Layer1={vars_epoch[0]:.4f}, Layer2={vars_epoch[1]:.4f}, Layer3={vars_epoch[2]:.4f}, Layer4={vars_epoch[3]:.4f}")

# 对比可视化：无 Normalization vs 添加 Batch Normalization 的每层特征方差对比
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 无 Normalization 对比图
for layer_idx in range(4):
    vars_layer = [epoch_vars[layer_idx] for epoch_vars in variances_nobn_history]
    axes[0].plot(range(1, epochs + 1), vars_layer, marker='o', linewidth=2, label=f'Layer {layer_idx+1}')
axes[0].set_xlabel('Epoch（轮数）')
axes[0].set_ylabel('Feature Variance（特征方差）')
axes[0].set_title('无 Normalization：特征方差衰减/不稳定')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

# 添加 Batch Normalization 对比图
for layer_idx in range(4):
    vars_layer = [epoch_vars[layer_idx] for epoch_vars in variances_bn_history]
    axes[1].plot(range(1, epochs + 1), vars_layer, marker='s', linewidth=2, label=f'Layer {layer_idx+1}')
axes[1].set_xlabel('Epoch（轮数）')
axes[1].set_ylabel('Feature Variance（特征方差）')
axes[1].set_title('添加 Batch Normalization：特征方差稳定控制')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()